# Тема 9 — Docker для LLM inference: контейнер как производственная единица

## Образ — это чертёж передвижной кухни, флот — это парк кухонь

**Что случилось в Topic 08.** Вы собрали шлюз (gateway) с честным контрактом: валидация,
ограниченная очередь с 429, таймауты. Но сервис жил в «works on my machine» земле: какой
Python — такой хост и дал, pip-установки на лету, cwd решает всё. Тема 9 делает из этого
сервиса **производственную единицу**: собрать быстро, собрать маленькой, пережить падение
и подняться снова — на обычном ноутбуке, без GPU.

**Четыре вопроса этой лабы (и один инцидент):**

1. **Введение.** Что такое образ, слой и контейнер — и как за пять минут запустить первый.
2. **Скорость сборки.** Почему правильный порядок слоёв и кэш компилятора сокращают
   пересборку с минут до секунд — измеряем на РЕАЛЬНОЙ компиляции llama.cpp.
3. **Размер.** Из чего складывается размер образа; почему веса модели доминируют; bake,
   mount или download — три способа доставки весов с тремя разными ценами.
4. **Прод-инцидент.** Под с заниженным лимитом памяти: OOMKill, crash-loop с растущим
   backoff, диагностика по журналу и коду выхода, починка сайзингом и
   **рестарт-до-готовности** как сумма именованных членов. Так работают поды в
   Kubernetes — словарь введём здесь, сам k8s будет в Topic 11.

**Правила игры.** (1) Все docker-клетки охраняются: без локального демона они печатают
причину пропуска и ничего не делают — валидация остаётся зелёной. (2) Секунды и мегабайты —
ВАШИ измерения на вашей машине; «эталонные записи курса» в тексте помечены явно и не
заменяют ваши. (3) В лабе работает НАСТОЯЩАЯ крошечная LLM: Qwen2.5-0.5B-Instruct
(Q4_K_M, 491 МБ) через llama.cpp `llama-server` — OpenAI-совместимый контракт из
Topics 04/08 в живом виде. (4) Предсказание замораживается до измерения — как в
Topics 06–07.

Рантайм: **локальный Docker**, CPU-образы для всех. Бюджет чистого времени: **~120 минут**
(2 контактных часа + половина самостоятельной); самые долгие ожидания — первая загрузка
весов (1–5 мин) и холодная сборка из исходников (2–15 мин в зависимости от машины,
один раз).

## ⚙ Как работать с лабой: локальный Docker (прочитайте до первой docker-клетки)

Рантайм этой лабы — ВАШ локальный Docker: Docker Desktop на macOS/Windows или Docker
Engine на Linux. Крошечная LLM в контейнере — CPU-only: ей не нужен GPU, но нужен запас
памяти. Jupyter — локальный, не Colab: клетки пишут файлы в `build/` (контекст сборки),
`models-cache/` (веса) и `results/` (журнал) рядом с ноутбуком.

**Локальный Python для ноутбука.** Ноутбук исполняется ВАШИМ локальным Python 3.10+:
запустите его как `jupyter lab` (или в VS Code) и один раз поставьте зависимости:
`%pip install tqdm pandas matplotlib` — прямо первой клеткой после setup, либо в
терминале. На Colab эти пакеты предустановлены, но docker-клетки там честно уйдут в skip.

**Проверка готовности — одна команда в терминале:**

```bash
docker info
```

Ответ пришёл — демон жив. Требования: `docker` на PATH, демон запущен, ~3 GiB свободного
диска под образы и веса, **Docker Desktop memory ≥ 4 GiB** (настройки → Resources; нашей
флот-симуляции из трёх реплик хватает и 4, комфортно — 6).

**Установка Docker (один раз, до лабы).** macOS и Windows:
<https://docs.docker.com/desktop/> — Docker Desktop (после установки запустите
приложение: оно и поднимает демон; на Windows понадобится WSL2 — установщик предложит
сам). Linux: <https://docs.docker.com/engine/install/> — Docker Engine под ваш
дистрибутив. Первый smoke-тест «живого» Docker будет первой клеткой лабы:
`docker run hello-world`.

**Docker в двух словах (если вы встречаете его впервые).** Образ — неизменяемый чертёж:
файловая система приложения вместе с зависимостями, сложенная слоями. Контейнер — обычный
процесс на ВАШЕЙ машине, запущенный по этому чертежу в изоляции: своё файловое
пространство, своё имя, ограниченные ресурсы — но то же ядро Linux, никакой «маленькой
виртуальной машины». Отсюда скорость старта, которую мы меряем в финальном инциденте.

**Производственный словарь этой лабы (мост к Kubernetes, Topic 11).** pod ≈ контейнер
(в этой лабе — ровно он); `OOMKilled` — статус контейнера, убитого лимитом памяти
(exit 137); crash-loop с растущей задержкой перезапуска ≈ `CrashLoopBackOff` в k8s;
лимиты `--memory` ≈ `resources.limits`; healthcheck ≈ liveness/readiness-пробы. Сегодня
мы смотрим эти механики на голом Docker, чтобы в k8s читать их как родные.

**Что будет, если демона нет.** Каждая docker-клетка охраняется проверкой `docker info`
с коротким таймаутом (свёрнутый хелпер `require_docker()`). Без демона клетка печатает
`skipped: локальный Docker недоступен — выполните этот шаг на своей машине (см. гид в
начале)` и больше ничего не делает. Поэтому валидация ноутбука (nbclient без Docker)
остаётся зелёной, а план-режим — предсказания, записи стенограмм, графики, журналы —
исполняется везде.

**GPU — не в этой лабе.** Прод-вариант образа (CUDA base + vLLM pin + `--gpus`) остаётся
демонстрацией владельца. Студенческий путь — крошечная CPU-LLM: он собирается и отвечает
на любой машине, где есть Docker, и честно показывает все прод-паттерны: очередь, лимиты,
OOM, рестарт.

In [ ]:
import json, os, subprocess, time, hashlib, urllib.error, urllib.request
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm  # правило курса: видимый прогресс в каждой клетке дольше ~5 с
from IPython.display import display

PROFILE = 'local'  # лаба исполняется на локальном Docker; без демона — план-режим
ATTEMPT = 'attempt-001'
RESULTS = os.path.join('results', PROFILE, ATTEMPT)
BUILD = 'build'            # контекст сборки: Dockerfile-ы, .dockerignore
MODELS = 'models-cache'    # веса модели (491 МБ) — вне контекста сборки, монтируются в контейнер
LLAMACPP_SHA = 'fd27b2034905f8dadd8bd6d2cc44595f4929d391'  # пин исходников = сборка образа-базы курса
os.makedirs(RESULTS, exist_ok=True)
os.makedirs(os.path.join(BUILD, 'app'), exist_ok=True)
os.makedirs(MODELS, exist_ok=True)
print(f'profile={PROFILE}  журнал={RESULTS}  контекст={BUILD}/  веса={MODELS}/')

In [ ]:
DOCKER_SKIP = ('skipped: локальный Docker недоступен — '
               'выполните этот шаг на своей машине (см. гид в начале)')

def docker(*args: str) -> tuple[int, str]:
    """Run one docker CLI command and return (returncode, combined output).

    Parameters
    ----------
    *args : str
        Arguments after the verb, e.g. docker('build', '-f', 'build/Dockerfile', 'build').

    Returns
    -------
    tuple of (int, str)
        Process return code and the combined stdout+stderr tail (last 4000 chars).
    """
    proc = subprocess.run(['docker', *args], capture_output=True, text=True, timeout=3600)
    return proc.returncode, ((proc.stdout + proc.stderr)[-4000:])

def require_docker(timeout_s: float = 10.0) -> bool:
    """Probe the local docker daemon with a short-timeout `docker info`.

    Parameters
    ----------
    timeout_s : float, default 10.0
        Wall budget for the probe; a missing binary also means False.

    Returns
    -------
    bool
        True only if the daemon answered successfully.
    """
    try:
        proc = subprocess.run(['docker', 'info'], capture_output=True, text=True, timeout=timeout_s)
    except (FileNotFoundError, subprocess.TimeoutExpired, OSError):
        return False
    return proc.returncode == 0

def wait_healthy(url: str, deadline_s: float = 120.0) -> float:
    """Poll one HTTP endpoint until it answers 200, with a visible progress bar.

    Parameters
    ----------
    url : str
        Health endpoint, e.g. 'http://127.0.0.1:8099/health'.
    deadline_s : float, default 120.0
        Wall budget before giving up.

    Returns
    -------
    float
        Waited seconds until the first 200 answer.

    Raises
    ------
    AssertionError
        If the endpoint never answered within the budget.
    """
    started = time.monotonic()
    with tqdm(total=deadline_s, desc='ждём /health', unit=' такт0.5с') as bar:
        while True:
            try:
                with urllib.request.urlopen(url, timeout=3) as resp:
                    if resp.status == 200:
                        return round(time.monotonic() - started, 1)
            except (urllib.error.URLError, OSError):
                pass
            assert time.monotonic() - started < deadline_s, f'{url} не ответил за {deadline_s} с'
            time.sleep(0.5)
            bar.update(0.5)

## Блок 0 — Первый контейнер и образ = упорядоченные слои

Каждая инструкция Dockerfile создаёт слой (layer): дельту над предыдущим. Образ — стопка
слоёв в порядке записи. Кухня собирается снизу вверх: фундамент (base image), коммуникации
(системные библиотеки и компиляторы), оборудование (зависимости) и только потом ваши
инструменты (код).

**Механизм build cache** — главная идея блока: cache hit для слоя возможен только если ни
он сам, ни один слой ПОД ним не изменились; правка слоя перестраивает его сам и всё ВЫШЕ.
Отсюда таблица порядка:

| Правка | Что пересоберётся | Почему |
|---|---|---|
| файл исходника | слой компиляции и выше | всё ниже не менялось |
| смена пина исходников | fetch-слой и выше | git fetch увидел другой коммит |
| тег/дайджест базового образа | вся стопка | фундамент сменился |

**Вопрос для предсказания ДО сборки.** Какой из трёх порядков инструкций даёт cache hit
слоя зависимостей при каждой правке кода?

- A: `COPY . .` → `RUN pip install -r requirements.txt`
- B: `COPY requirements.txt …` → `RUN pip install …` → `COPY app/ app/`
- C: `RUN pip install …` → `COPY . .`

Заморозьте выбор и предсказания в `build-plan.json` (клетка ниже) ДО первой сборки — как
прогноз колена в Topic 06. Сюда же входят **члены рестарта**: из чего сложится время
«упал → снова отвечает» — этот план Блок 4 проверит измерением.

In [ ]:
if not require_docker():
    print(DOCKER_SKIP)
else:
    # Оценка времени: ~5–30 с при первом прогоне (тянется крошечный образ hello-world).
    rc, out = docker('run', '--rm', 'hello-world')
    _banner = [l.strip() for l in out.splitlines() if 'Hello from Docker' in l]
    _rc, ver_out = docker('version', '--format', '{{.ServerVersion}} ({{.OSType}}/{{.Architecture}})')
    _rc, images_out = docker('images', '--format', '{{.Repository}}:{{.Tag}} {{.Size}}', 'hello-world')
    print(f'первый артефакт: hello-world rc={rc}; {_banner[0] if _banner else "баннер в журнале клетки"}; '
          f'демон: {ver_out.strip()}; образ: {images_out.strip() or "не найден"}')

In [ ]:
def plan_cache_order() -> dict:
    """Freeze the build plan BEFORE the first build: layer order and cache predictions.

    Returns
    -------
    dict
        chosen_order ('A'/'B'/'C'), per-edit invalidation predictions, restart terms
        and the sizing note — all frozen into build-plan.json by the caller.
    """
    options = {
        'A': ['COPY . .', 'RUN pip install -r requirements.txt'],
        'B': ['COPY requirements.txt requirements.txt',
              'RUN pip install -r requirements.txt',
              'COPY app/ app/'],
        'C': ['RUN pip install -r requirements.txt', 'COPY . .'],
    }
    # Уровень A: почти написано. Выберите порядок слоёв — впишите одну букву.
    chosen_order = 'TODO'  # 'A', 'B' или 'C' из options выше
    return dict(
        chosen_order=chosen_order,
        options=options,
        predictions=dict(
            edit_source='TODO: какие слои пересоберутся и почему?',
            change_source_pin='TODO: какие слои пересоберутся и почему?',
            change_base='TODO: какие слои пересоберутся и почему?'),
        restart_terms=dict(
            cold='TODO: члены времени "упал → отвечает" через + (подсказка: extract, weights, init, health)',
            dominant='TODO: какой член ожидаете доминирующим для 491 МБ весов и почему'),
        why='TODO: одно предложение — почему выбранный порядок бережёт кэш')

_plan = plan_cache_order()
with open(os.path.join(RESULTS, 'build-plan.json'), 'w', encoding='utf-8') as fh:
    json.dump(_plan, fh, ensure_ascii=False, indent=1)
print(f"build-plan.json заморожен ДО первой сборки: порядок слоёв = {_plan['chosen_order']}")

## Блок 1 — Скорость сборки: два кэша на одной реальной компиляции

Компиляция llama.cpp из исходников — минуты даже на быстрой машине. Мы измерим, как её
собирают в проде, и увидим **два разных кэша**, которые часто путают:

1. **Кэш слоёв (layer cache).** Инструкция Dockerfile не выполняется повторно, пока не
   изменились её входы. Слой инструментов (`apt-get install …`) кэшируется навсегда; слой
   `git fetch` pinned-коммита — пока пин не сменился; слой компиляции — при любой правке
   исходников. Пересборка без изменений: **секунды**.
2. **Кэш содержимого (cache mount).** `RUN --mount=type=cache,target=/root/.cache/ccache`
   переживает инвалидацию слоя компиляции: ccache помнит объектные файлы, и пересборка
   после правки одной строки занимает **единицы минут вместо полной компиляции**. Это кэш
   СОДЕРЖИМОГО, а не инструкций — поэтому он переживает то, что слой-кэш не переживёт.

**Ловушка слоя инструментов, пойманная в этой лабе по-настоящему.** Первая версия
Dockerfile не ставила `ca-certificates` — и сборка падала на `git fetch` с
`server certificate verification failed. CAfile: none`. Почему apt при этом работал?
Он ходит по plain-HTTP к зеркалам Ubuntu, а GitHub требует TLS. Мораль: инструментальный
слой — это контракт с сетью, перечислите в нём всё, до чего дотянется сборка.

**План эксперимента (замеры — ВАШИ):** (1) холодная сборка с пустым кэшем ccache;
(2) пересборка без изменений — проверка layer-кэша; (3) правка одной строки исходников при
тёплом ccache — проверка cache mount. Эталонные записи курса — в таблице после клеток.
Оценка времени: холодная сборка **2–15 мин** (прогресс печатается каждые 15 с), остальные —
секунды и единицы минут. Не хотите ждать холодную? Пропустите клетку и подставьте записи
курса в сверке — но тогда ваш журнал не будет содержать ваших измерений, и это честно
видно по файлу build-timings.json.

In [ ]:
BUILDER_LINES = [
    '# v-builder: собирает llama-server из pinned-исходников; инструменты остаются в builder.',
    'FROM ubuntu:24.04 AS build',
    'ARG PATCH_COMMENT=""',
    'RUN apt-get update && apt-get install -y --no-install-recommends \\',
    '      build-essential cmake git ccache ca-certificates libcurl4-openssl-dev \\',
    '    && rm -rf /var/lib/apt/lists/*',
    'WORKDIR /src',
    '# fetch-слой: кэшируется, пока пин коммита не меняется',
    'RUN git init -q . \\',
    '    && git remote add origin https://github.com/ggml-org/llama.cpp \\',
    '    && git fetch -q --depth 1 origin __LLAMACPP_SHA__ \\',
    '    && git checkout -q --detach FETCH_HEAD',
    '# compile-слой: инвалидируется при правке исходников; ccache-маунт переживает это',
    'RUN --mount=type=cache,target=/root/.cache/ccache,sharing=locked \\',
    '    if [ -n "$PATCH_COMMENT" ]; then echo "$PATCH_COMMENT" >> src/llama.cpp; fi \\',
    '    && cmake -B build -DGGML_NATIVE=ON -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release \\',
    '             -DCMAKE_C_COMPILER_LAUNCHER=ccache -DCMAKE_CXX_COMPILER_LAUNCHER=ccache \\',
    '    && cmake --build build --config Release -j"$(nproc)" --target llama-server',
    '',
    'FROM ubuntu:24.04',
    'RUN apt-get update && apt-get install -y --no-install-recommends libgomp1 curl \\',
    '    && rm -rf /var/lib/apt/lists/* && useradd --create-home appuser',
    'COPY --from=build /src/build/bin/llama-server /app/llama-server',
    'WORKDIR /app',
    'USER appuser',
    'EXPOSE 8080',
    'HEALTHCHECK --interval=5s --timeout=3s --start-period=10s \\',
    '  CMD curl -fsS http://127.0.0.1:8080/health || exit 1',
    'ENTRYPOINT ["/app/llama-server"]',
]
BUILDER_DOCKERFILE = '\n'.join(BUILDER_LINES).replace('__LLAMACPP_SHA__', LLAMACPP_SHA)

# Уровень B: скелет Dockerfile дан целиком — найдите три ответа и впишите их.
BLOCK1_ANSWERS = dict(
    tool_layer='TODO',   # какие три пакета инструментального слоя нужны сборке из сети?
                         # (подсказка: компилятор-набор, система сборки, TLS-сертификаты)
    fetch_layer='TODO',  # что кэширует fetch-слой и когда он инвалидируется?
    cache_mount='TODO',  # какой mount в compile-слое переживает его инвалидацию и почему?
)

_missing = [k for k, v in BLOCK1_ANSWERS.items() if str(v).startswith('TODO')]
if _missing:
    print(f'заполните ответы блока 1: {_missing} — Dockerfile.builder не записан')
else:
    (Path(BUILD) / 'Dockerfile.builder').write_text(BUILDER_DOCKERFILE, encoding='utf-8')
    print(f'Dockerfile.builder записан: {len(BUILDER_DOCKERFILE.splitlines())} строк — '
          f'multi-stage, pinned fetch, ccache cache-mount')

In [ ]:
if not require_docker():
    print(DOCKER_SKIP)
else:
    # Оценка времени: 2–15 мин холодной компиляции; прогресс — tqdm.
    started = time.monotonic()
    log_path = os.path.join(BUILD, 'builder-cold.log')
    with open(log_path, 'w', encoding='utf-8') as log_fh:
        proc = subprocess.Popen(['docker', 'build', '-f', 'build/Dockerfile.builder',
                                 '-t', 't09-builder:cold', BUILD],
                                stdout=log_fh, stderr=subprocess.STDOUT)
        with tqdm(total=900, desc='docker build (cold)', unit=' с') as bar:
            while proc.poll() is None:
                time.sleep(5)
                bar.update(5)
    rc = proc.returncode
    cold_s = round(time.monotonic() - started, 1)
    timings = dict(cold_s=cold_s if rc == 0 else None)
    with open(os.path.join(RESULTS, 'build-timings.json'), 'w', encoding='utf-8') as fh:
        json.dump(timings, fh, ensure_ascii=False, indent=1)
    _tail = open(log_path, encoding='utf-8').read()[-200:] if rc != 0 else ''
    print(f'холодная сборка: rc={rc} за {cold_s} с → build-timings.json' + (_tail if rc != 0 else ''))

In [ ]:
if not require_docker():
    print(DOCKER_SKIP)
else:
    _path = os.path.join(RESULTS, 'build-timings.json')
    if not os.path.exists(_path):
        print('нет build-timings.json: выполните клетку холодной сборки (или примите, '
              'что этот журнал без ваших измерений)')
    else:
        # Оценка времени: пересборка без правок ~1–20 с; правка строки с тёплым ccache —
        # единицы минут. Прогресс — tqdm по двум сборкам.
        with open(_path, encoding='utf-8') as fh:
            timings = json.load(fh)
        rows = []
        for label, extra in tqdm((('layer_cache', []),
                                  ('ccache_hit', ['--build-arg',
                                                  'PATCH_COMMENT=// t09-cache-drill'])),
                                 desc='пересборки', unit=' сборка'):
            t0 = time.monotonic()
            rc, out = docker('build', '-q', *extra, '-f', 'build/Dockerfile.builder',
                             '-t', f't09-builder:{label}', BUILD)
            wall = round(time.monotonic() - t0, 1) if rc == 0 else None
            timings['layer_cache_s' if label == 'layer_cache' else 'ccache_hit_s'] = wall
            rows.append(dict(build=label, rc=rc, wall_s=wall))
        with open(_path, 'w', encoding='utf-8') as fh:
            json.dump(timings, fh, ensure_ascii=False, indent=1)
        rows.append(dict(build='cold (клетка выше)', rc=0, wall_s=timings.get('cold_s')))
        display(pd.DataFrame(rows).style.set_caption(
            'три сборки: ваши измерения (build-timings.json) — cold, layer-cache, ccache-hit'))

### Эталонные записи курса — сравните со своими числами

Измерено на эталонной машине курса (MacBook, Docker Desktop, VM ~3.8 GiB / 10 vCPU,
7 ноября 2026). Это **записи, не ваше измерение**: расхождение в разы — нормально и
интересно (другой CPU, другой диск, другая скорость сети).

| Сценарий | Эталонная запись |
|---|---|
| холодная компиляция llama.cpp (10 vCPU, пустой ccache) | 142 с |
| пересборка без правок (layer cache) | 0.4 с |
| правка одной строки, тёплый ccache | 44 с |
| from-source runtime-образ (только llama-server, без инструментов) | 150 MB |
| образ-база llama.cpp:server (распакованный) | 1.18 GB |
| публикуемый vllm/vllm-openai:latest (сжатый, запись Docker Hub 02.11.2026) | ~10.4 GB |

Вопрос для сверки: у вас layer-cache и ccache-эффекты сохранили **порядок** (доли секунды
против минуты против полной компиляции) или соотношение другое? Что это говорит о
переносимости таких «ускорений»?

## Блок 2 — Размер образа: веса доминируют, доставка весов — выбор цены

Размер образа = сумма его слоёв (`docker history` — бухгалтерия). Для LLM-образа
бухгалтерия жёсткая: бинарник llama-server — единицы мегабайт, веса Qwen2.5-0.5B Q4_K_M —
**491 400 032 байта**. Что бы вы ни делали с Dockerfile, размеры диктуют веса — вопрос не
«как уменьшить», а **какую цену вы выбираете** за способ доставки весов:

| Способ | Размер образа | Цена | Когда выбирают |
|---|---|---|---|
| **bake** (COPY в образ) | +491 МБ | тяжёлый pull, лёгкий старт | образ самодостаточен, реестр близко |
| **mount** (`-v` с хоста) | +0 МБ | зависимость от хоста | один хост, парк реплик на машине |
| **download** (сеть на старте) | +0 МБ | доминирующий член рестарта | эфемерные узлы, общий кэш |

Для контраста (эталонные записи курса): наш from-source runtime-образ — **150 MB**;
образ-база llama.cpp:server — **1.18 GB** распакованного; публикуемый vllm/vllm-openai —
**~10.4 GB сжатого** (полный стек torch + CUDA). Мы собираем на базе апстрима, веса в
образ НЕ запекаем и получаем единицу, которую тянет любой ноутбук.

**Пин по дайджесту.** Тег `:server` плавает; дайджест `sha256:…` — это точные байты.
Строка `FROM ghcr.io/ggml-org/llama.cpp@sha256:6d581c78…` (клетка ниже) гарантирует, что
«вчера работало» останется «сегодня работает». Да, флаги тоже дрейфуют: в свежих сборках
llama.cpp `--mlock`/`--no-mmap` заменены на `--load-mode …` — ещё один аргумент за пин.

Оценка времени клеток: сборки тонкого образа — секунды (мета-слои), history —
миллисекунды, bake-вариант — десятки секунд (копия весов).

In [ ]:
BASE_DIGEST = 'sha256:6d581c788abba2dfa23ff78dba37ea0abfc7d838c49bcc98f4be2103df9772b3'

THIN_V1 = '\n'.join([
    '# v1: lean, digest-pinned, non-root, healthchecked (веса НЕ запечены)',
    'FROM ghcr.io/ggml-org/llama.cpp@' + BASE_DIGEST,
    'LABEL org.opencontainers.image.title="t09-gw v1 (lean)"',
    'HEALTHCHECK --interval=5s --timeout=3s --start-period=10s \\',
    '  CMD curl -fsS http://127.0.0.1:8080/health || exit 1',
    'USER 1000:1000',
])
THIN_V0 = '\n'.join([
    '# v0: works on my machine — плавающий тег, root, без healthcheck',
    'FROM ghcr.io/ggml-org/llama.cpp:server',
    'LABEL org.opencontainers.image.title="t09-gw v0 (naive)"',
])

def write_thin_dockerfiles() -> dict:
    """Write the v0 and v1 thin Dockerfiles into the build context.

    Returns
    -------
    dict
        Relative paths mapped to file sizes in bytes.
    """
    # Уровень B: скелет готов — заполните шаги.
    # Шаг 1: соберите словарь {'Dockerfile': THIN_V1, 'Dockerfile.v0': THIN_V0}
    # Шаг 2: каждый текст запишите в Path(BUILD) / имя, encoding='utf-8'
    # Шаг 3: верните {имя: path.stat().st_size}
    raise NotImplementedError('TODO: запишите оба thin-Dockerfile по шагам')

_staged = write_thin_dockerfiles()
print('thin-Dockerfiles записаны: ' + ', '.join(f'{k} ({v} B)' for k, v in _staged.items()))

In [ ]:
if not require_docker():
    print(DOCKER_SKIP)
else:
    # Оценка времени: ~10–60 с при первом прогоне (тянется база), дальше секунды.
    started = time.monotonic()
    rc0, out0 = docker('build', '-q', '-f', 'build/Dockerfile.v0', '-t', 't09-gw:v0', BUILD)
    t0 = round(time.monotonic() - started, 1)
    rc1, out1 = docker('build', '-q', '-f', 'build/Dockerfile', '-t', 't09-gw:v1', BUILD)
    t1 = round(time.monotonic() - started, 1) - t0
    _rc, images_out = docker('images', '--format', '{{.Repository}}:{{.Tag}} {{.Size}}', 't09-gw')
    print(f'v0 (плавающий тег): rc={rc0} за {t0} с; v1 (дайджест): rc={rc1} за {t1} с; '
          'размеры идентичны — v0→v1 отличаются ТОЛЬКО мета-слоями (0 B), '
          'пин и healthcheck не весят ничего, но меняют всё:\n' + images_out.strip())

In [ ]:
def summarize_layers(history_text: str) -> list:
    """Itemize docker history output into per-layer rows for size accounting.

    Parameters
    ----------
    history_text : str
        Lines of `docker history --format SIZE,TAB,CreatedBy`, newest layer first.

    Returns
    -------
    list of dict
        Rows: size_mb (float, 0.0 for metadata layers), instruction (short).
    """
    # Уровень B: скелет готов.
    # Шаг 1: разберите text на строки; каждую — по табуляции на размер и инструкцию.
    # Шаг 2: размер приведите к MB, единицы РЕГИСТРОНЕЗАВИСИМО (docker пишет 'kB'):
    #        '1.18GB' → 1180.0, '452kB' → 0.452, '0B' → 0.0.
    # Шаг 3: инструкцию обрежьте до ~70 символов; пустую замените на '(metadata layer)'.
    # Шаг 4: верните список dict(size_mb=..., instruction=...).
    raise NotImplementedError('TODO: разберите вывод docker history по шагам')

if not require_docker():
    print(DOCKER_SKIP)
else:
    _rc, history_out = docker('history', '--format', '{{.Size}}\t{{.CreatedBy}}', 't09-gw:v1')
    _layer_rows = summarize_layers(history_out)
    with open(os.path.join(RESULTS, 'size-accounting.json'), 'w', encoding='utf-8') as fh:
        json.dump(dict(layers=_layer_rows), fh, ensure_ascii=False, indent=1)
    display(pd.DataFrame(_layer_rows).head(10).style.set_caption(
        'ведомость слоёв t09-gw:v1 (docker history) — ваши измерения (size-accounting.json)'))

In [ ]:
BAKE_ASKED = False  # поменяйте на True и перезапустите клетку: измерьте цену bake
# Оценка времени при BAKE_ASKED=True: ~10–40 с (копия 491 МБ весов в контекст + слой).
if not BAKE_ASKED:
    print('bake-вариант пропущен по вашему выбору (BAKE_ASKED=False). '
          'Эталонная запись курса: COPY весов добавляет слой ~491 MB, '
          'образ 1.18 GB → ~1.67 GB. Поменяйте BAKE_ASKED и перезапустите, чтобы измерить.')
elif not require_docker():
    print(DOCKER_SKIP)
else:
    _src = Path(MODELS) / 'qwen2.5-0.5b-instruct-q4_k_m.gguf'
    assert _src.exists(), 'весов нет: выполните клетку загрузки из Блока 3'
    (Path(BUILD) / 'model.gguf').write_bytes(_src.read_bytes())
    (Path(BUILD) / 'Dockerfile.baked').write_text(
        'FROM t09-gw:v1\nCOPY model.gguf /models/model.gguf\n', encoding='utf-8')
    t0 = time.monotonic()
    rc, out = docker('build', '-q', '-f', 'build/Dockerfile.baked', '-t', 't09-gw:baked', BUILD)
    _rc, images_out = docker('images', '--format', '{{.Repository}}:{{.Tag}} {{.Size}}', 't09-gw')
    print(f'bake-сборка: rc={rc} за {time.monotonic() - t0:.0f} с; цена bake видна в размерах:\n'
          + images_out.strip())
    docker('rmi', '-f', 't09-gw:baked')
    (Path(BUILD) / 'model.gguf').unlink(missing_ok=True)

## Блок 3 — Контракт через границу: та же LLM, другой мир

В Topics 04/08 контракт курса выглядел так: `/health` → 200, completion → 200 с выбором и
usage. Ничего в этом контракте не меняется от того, что сервер живёт в контейнере — в
этом и смысл образа: поведение больше не зависит от хоста. Проба-верификация ниже
отправляет РЕАЛЬНЫЙ запрос к РЕАЛЬНОЙ крошечной LLM через границу:
`localhost:порт → контейнер → llama-server → Qwen2.5-0.5B`.

**Три обязательства перед запуском:**

1. **Веса.** Скачиваем один раз в `models-cache/` (491 МБ, прогресс и оценка времени —
   в клетке). Контейнеру отдаём монтированием `-v`: способ доставки «mount» из таблицы
   Блока 2.
2. **Порты.** `-p host:container` — единственная дорога из ноутбука в контейнер.
3. **Лимит.** `--memory` — стена, за которой живёт под; в Блоке 4 мы увидим, что бывает,
   когда стена ниже резидентной памяти.

Оценка времени: загрузка весов 1–5 мин (один раз, кэш дальше), запуск и пробы — секунды.

In [ ]:
MODEL_URL = ('https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF/resolve/main/'
             'qwen2.5-0.5b-instruct-q4_k_m.gguf')
MODEL_FILE = os.path.join(MODELS, 'qwen2.5-0.5b-instruct-q4_k_m.gguf')
MODEL_BYTES = 491_400_032  # объявленный размер; сверим по факту

if os.path.exists(MODEL_FILE) and os.path.getsize(MODEL_FILE) == MODEL_BYTES:
    print(f'веса уже в кэше: {MODEL_FILE} ({MODEL_BYTES} B) — загрузка не повторяется')
else:
    # Оценка времени: 1–5 мин при первом прогоне (491 МБ); прогресс — по чанкам 1 МБ.
    started = time.monotonic()
    req = urllib.request.Request(MODEL_URL)
    with urllib.request.urlopen(req, timeout=60) as resp, \
            open(MODEL_FILE, 'wb') as fh, \
            tqdm(total=MODEL_BYTES, desc='загрузка весов', unit=' B', unit_scale=True) as bar:
        while True:
            chunk = resp.read(1 << 20)
            if not chunk:
                break
            fh.write(chunk)
            bar.update(len(chunk))
    got = os.path.getsize(MODEL_FILE)
    print(f'веса загружены за {time.monotonic() - started:.0f} с: {got} B '
          f'(объявлено {MODEL_BYTES} B) — {"совпало" if got == MODEL_BYTES else "РАСХОЖДЕНИЕ"}')

In [ ]:
RUN_FLAGS = dict(
    name='t09-gw',
    port_host='TODO',       # впишите свободный порт хоста, например 8099
    memory_limit='TODO',    # впишите лимит с запасом, например 1500m
    model_mount='TODO',     # впишите монтирование весов, например MODEL_FILE:/models/model.gguf:ro
)

def run_command(flags: dict, image: str = 't09-gw:v1') -> list:
    """Assemble the `docker run` argument list from the chosen flags.

    Parameters
    ----------
    flags : dict
        RUN_FLAGS after filling: name, port_host, memory_limit, model_mount.
    image : str, default 't09-gw:v1'
        Image to run.

    Returns
    -------
    list of str
        Arguments after `docker`: run -d --name … -p … --memory … -v … IMAGE
        plus the llama-server arguments (model, host, port, context 1024).
    """
    # Уровень B: скелет готов — соберите аргументы по шагам.
    # Шаг 1: ['run', '-d', '--name', flags['name'], '-p', f"{flags['port_host']}:8080"]
    # Шаг 2: '--memory', flags['memory_limit']
    # Шаг 3: '-v', flags['model_mount']
    # Шаг 4: image + ['-m', '/models/model.gguf', '--host', '0.0.0.0', '--port', '8080',
    #        '-c', '1024', '-ngl', '0']
    raise NotImplementedError('TODO: соберите команду запуска по шагам')

if not require_docker():
    print(DOCKER_SKIP)
else:
    _todo = [k for k, v in RUN_FLAGS.items() if str(v).startswith('TODO')]
    if _todo:
        print(f'заполните RUN_FLAGS: {_todo} — docker run не выполнялся')
    else:
        cmd = run_command(RUN_FLAGS)
        docker('rm', '-f', RUN_FLAGS['name'])
        rc, out = docker(*cmd)
        waited = wait_healthy(f"http://127.0.0.1:{RUN_FLAGS['port_host']}/health", deadline_s=90.0)
        print(f"контейнер {RUN_FLAGS['name']} запущен (rc={rc}), "
              f"/health через границу за {waited} c — LLM жива в контейнере")

In [ ]:
def probe_openai_contract(base_url: str, timeout_s: float = 60.0) -> list:
    """Probe the OpenAI-style contract across the container boundary.

    Parameters
    ----------
    base_url : str
        Server root, e.g. 'http://127.0.0.1:8099'.
    timeout_s : float, default 60.0
        Wall budget for the completion request.

    Returns
    -------
    list of dict
        One row per probe: probe, expected, got, ok.
    """
    # Уровень B: скелет готов.
    # Шаг 1: GET base_url + '/health' → ждём status 200.
    # Шаг 2: POST base_url + '/v1/chat/completions' с {"messages":[{"role":"user",
    #        "content":"Reply with the single word: OK"}], "max_tokens": 8,
    #        "temperature": 0} → ждём 200 и непустой choices[0].message.content.
    # Шаг 3: из того же ответа возьмите usage.total_tokens (число > 0).
    # Шаг 4: верните строки dict(probe=..., expected=..., got=..., ok=...).
    raise NotImplementedError('TODO: реализуйте три зонда контракта')

if not require_docker():
    print(DOCKER_SKIP)
else:
    _rows = probe_openai_contract(f"http://127.0.0.1:{RUN_FLAGS['port_host']}")
    with open(os.path.join(RESULTS, 'contract-journal.json'), 'w', encoding='utf-8') as fh:
        json.dump(_rows, fh, ensure_ascii=False, indent=1)
    display(pd.DataFrame(_rows).style.set_caption(
        'OpenAI-контракт через границу контейнера — ваши измерения (contract-journal.json)'))

## Блок 4 — Прод-инцидент: под упал по памяти. Сделайте так, чтобы он вставал быстро.

Это кульминация лабы — точный сценарий продакшена, воспроизведённый на вашем ноутбуке:

1. **Предсказание ДО запуска.** Реплика с `-c 1024` ест сотни МБ резидентно (веса 491 МБ +
   KV-кэш + буферы). Ваш лимит `--memory 400m` ниже этого. Что сделает ядро, когда процесс
   коснётся страниц за стеной? Запишите предсказание в клетку запуска.
2. **Инцидент.** Под с заниженным лимитом и `--restart=on-failure`: OOMKill → рестарт →
   снова OOMKill → **crash-loop с растущим backoff**. В Kubernetes это называют
   `CrashLoopBackOff` (10 с → … → 5 мин); docker делает то же самое (100 мс → … → 1 мин).
   Мы снимем кривую рестартов по-настоящему.
3. **Диагностика.** Наружу видны только exit code 137 и хвост журнала: почему 137, а не 1?
   `docker inspect` ответит полем `OOMKilled=true`.
4. **Починка сайзингом и экономика рестарта.** Лимит = резидентная память с запасом.
   Дальше измеряем **рестарт-до-готовности**: extract слоёв + загрузка весов + init +
   первый `/health`. Вот здесь платится за решения Блока 2: маленький образ — быстрый
   extract, локальные веса — быстрая загрузка. Минимизация размера — это скорость
   восстановления.

Дисциплина разбора: причина → доказательство (код возврата + хвост лога + inspect) →
исправление → повторное измерение. Изменение без названного механизма — не улучшение,
а суеверие. Оценка времени: инцидент ~30 с, наблюдение crash-loop ~25 с, починка с двумя
рестартами ~40 с — всё с прогрессом.

In [ ]:
INCIDENT_PREDICTION = dict(
    what_happens='TODO: что случится с контейнером --memory 400m при весах 491 МБ?',
    exit_code='TODO: какой код выхода ожидаете и почему 137?',
)
# Оценка времени: ~40–60 с (запуск + наблюдение 20 с, сэмплирование каждые 2 с).
if not require_docker():
    print(DOCKER_SKIP)
else:
    _open = [k for k, v in INCIDENT_PREDICTION.items() if str(v).startswith('TODO')]
    if _open:
        print(f'заполните INCIDENT_PREDICTION: {_open} — инцидент не запускался '
              '(предсказание ДО измерения, как в Topics 06–07)')
    else:
        docker('rm', '-f', 't09-inc')
        docker('run', '-d', '--name', 't09-inc', '--restart', 'on-failure:20',
               '--memory', '400m', '--memory-swap', '400m',
               '-v', f'{Path(MODEL_FILE).resolve()}:/models/model.gguf:ro',
               't09-gw:v1', '-m', '/models/model.gguf', '--load-mode', 'mmap+mlock',
               '--host', '0.0.0.0', '--port', '8080', '-c', '1024', '-ngl', '0')
        timeline = []
        started = time.monotonic()
        # Наблюдаем 20 с: снимаем RestartCount каждые 2 с (tqdm — видимый прогресс).
        # Inspect читается через --format: компактный JSON вместо полного манифеста.
        def incident_state() -> tuple[dict, int]:
            """Read the incident container state and restart count via compact inspect formats.

            Returns
            -------
            tuple of (dict, int)
                The parsed `docker inspect .State` JSON object and the RestartCount.
            """
            _rc, state_out = docker('inspect', '--format', '{{json .State}}', 't09-inc')
            _rrc, count_out = docker('inspect', '--format', '{{.RestartCount}}', 't09-inc')
            return json.loads(state_out), int(count_out.strip() or 0)
        for _ in tqdm(range(10), desc='crash-loop', unit=' x2c'):
            time.sleep(2.0)
            try:
                _state, _count = incident_state()
                _row = dict(t_s=round(time.monotonic() - started, 1), restarts=_count)
            except Exception:
                _row = dict(t_s=round(time.monotonic() - started, 1), restarts=None)
            timeline.append(_row)
        _st, _count = incident_state()
        log_tail = docker('logs', '--tail', '3', 't09-inc')[1]
        incident = dict(prediction=INCIDENT_PREDICTION,
                        exit_code=_st.get('ExitCode'),
                        oomkilled=_st.get('OOMKilled'),
                        restarts=_count, timeline=timeline,
                        log_tail=log_tail[-300:])
        with open(os.path.join(RESULTS, 'oom-crashloop.json'), 'w', encoding='utf-8') as fh:
            json.dump(incident, fh, ensure_ascii=False, indent=1)
        docker('rm', '-f', 't09-inc')
        print(f"инцидент: exit={incident['exit_code']}, OOMKilled={incident['oomkilled']}, "
              f"рестартов за 20 с: {incident['restarts']} → oom-crashloop.json. "
              'Заметьте: интервалы рестартов РАСТУТ — это backoff, docker-версия '
              'CrashLoopBackOff.')

In [ ]:
def diagnose_exit(exit_code: int, log_tail: str, oomkilled: bool | None = None) -> dict:
    """Classify one container failure from its exit code, log tail and inspect flag.

    Parameters
    ----------
    exit_code : int
        Container process exit code (e.g. 137, 1, 125, 127).
    log_tail : str
        Last log lines of the failed container.
    oomkilled : bool or None, default None
        `docker inspect .State.OOMKilled`, когда инспекция доступна.

    Returns
    -------
    dict
        Keys cause, evidence, fix — the S5 record for the journal.
    """
    # Уровень C: контракт — docstring выше, тела нет.
    # Ожидаемые классы: 137/oomkilled=true → cgroup-лимит памяти; 1 + manifest unknown
    # → tag drift; 125 → ошибка демона; 127 → команда не найдена; остальное — не классифицировано.
    raise NotImplementedError('TODO: реализуйте классификацию по контракту')

# Учебные стенограммы прошлых прогонов (записи, не живые данные): разбор исполняется везде.
OOM_LOG = """srv load_model: loading model '/models/model.gguf'
kernel: oom-kill:memcg limit exceeded, victim llama-server killed by SIGKILL
exit status 137"""
PULL_LOG = """Error response from daemon: manifest for ghcr.io/ggml-org/llama.cpp:server-typo: not found
manifest unknown"""
_rows = [dict(drill='oom-400m (стенограмма)', **diagnose_exit(137, OOM_LOG, oomkilled=True)),
         dict(drill='tag-drift (стенограмма)', **diagnose_exit(1, PULL_LOG))]
_live_path = os.path.join(RESULTS, 'oom-crashloop.json')
if os.path.exists(_live_path):  # живой инцидент выше — добавляем реальные доказательства
    with open(_live_path, encoding='utf-8') as fh:
        _live = json.load(fh)
    _rows.append(dict(drill='oom-400m (живой инцидент)',
                      **diagnose_exit(_live['exit_code'], _live['log_tail'],
                                      oomkilled=_live['oomkilled'])))
display(pd.DataFrame(_rows).style.set_caption(
    'разбор: причина → доказательство → исправление (живой инцидент попал в журнал)'))

In [ ]:
SIZING = dict(
    resident_note='TODO: сколько ест реплика (docker stats из Блока 3 или эталонная запись ~760 MiB)?',
    new_limit='TODO: новый лимит с запасом (например, resident x1.5) — впишите как 1200m',
    why='TODO: названный механизм — почему этот лимит прекращает crash-loop',
)
# Оценка времени: ~40–60 с (запуск + два рестарта с прогрессом).
if not require_docker():
    print(DOCKER_SKIP)
else:
    _open = [k for k, v in SIZING.items() if str(v).startswith('TODO')]
    if _open:
        print(f'заполните SIZING: {_open} — починка не выполнялась '
              '(изменение без названного механизма — суеверие)')
    else:
        docker('rm', '-f', 't09-fix')
        fix_port = int(RUN_FLAGS['port_host']) + 1  # свой порт: не делить с репликой Блока 3
        docker('run', '-d', '--name', 't09-fix', '--restart', 'on-failure:5',
               '--memory', SIZING['new_limit'], '--memory-swap', SIZING['new_limit'],
               '-p', f'{fix_port}:8080',
               '-v', f'{Path(MODEL_FILE).resolve()}:/models/model.gguf:ro',
               't09-gw:v1', '-m', '/models/model.gguf',
               '--host', '0.0.0.0', '--port', '8080', '-c', '1024', '-ngl', '0')
        fix_url = f'http://127.0.0.1:{fix_port}/health'
        first = wait_healthy(fix_url, deadline_s=90.0)
        docker('stats', '--no-stream', 't09-fix')  # прогрев stats перед снятием
        stats = docker('stats', '--no-stream', '--format', '{{.MemUsage}}', 't09-fix')[1].strip()
        restarts_to_ready = []
        for i in tqdm(range(2), desc='рестарт-до-готовности', unit=' рестарт'):
            t0 = time.monotonic()
            docker('restart', 't09-fix')
            waited = wait_healthy(fix_url, deadline_s=90.0)
            restarts_to_ready.append(round(time.monotonic() - t0, 2))
            time.sleep(1.0)
        econ = dict(sizing=SIZING, first_ready_s=first, resident=stats,
                    restart_to_ready_s=restarts_to_ready)
        with open(os.path.join(RESULTS, 'restart-economics.json'), 'w', encoding='utf-8') as fh:
            json.dump(econ, fh, ensure_ascii=False, indent=1)
        docker('rm', '-f', 't09-fix')
        print(f'починка: resident={stats}, рестарт-до-готовности {restarts_to_ready} с '
              '→ restart-economics.json. Эталонная запись курса: ~1.1 с на тёплой машине — '
              'сравните и назовите, какие члены рестарта вы платите.')

## Stretch — SBOM и цепочка поставок

SBOM (software bill of materials) — инвентарь всего, что внутри образа. Один образ — один
документ. Три вопроса, которые он делает явными:

1. **Плавающий тег базы.** `:server` сегодня и через полгода — разные байты при одном теге;
   дайджест `sha256:…` из Блока 2 — противоядие.
2. **Зависимости без границ.** Пин сборки (LLAMACPP_SHA) и пин весов (имя файла + размер
   491 400 032 B) — версии с границами; всё остальное — лотерея.
3. **Слои всё помнят.** Файл, удалённый следующей инструкцией, остаётся в предыдущем слое:
   `docker history` читает то, что вы «стёрли». Секрет в слое — секрет навсегда.

Инструменты: `docker sbom` (плагин) или `docker scout sbom`. Клетка ниже сохранит
инвентарь в журнал; если инструмента нет — напечатает честную инструкцию и не сделает
вид, что всё прошло. Ограничение: один инструмент и один образ — это не сертификат
безопасности.

In [ ]:
SUPPLY_RISKS = dict(
    risk_1='TODO: плавающий тег базового образа — почему это риск цепочки поставок',
    risk_2='TODO: зависимости без точных версий (исходники, веса) — почему это риск',
    risk_3='TODO: что остаётся в истории слоёв после удаления файла/секрета',
)
if not require_docker():
    print(DOCKER_SKIP)
else:
    rc, out = docker('sbom', 't09-gw:v1')
    tool = 'docker sbom'
    if rc != 0:
        rc, out = docker('scout', 'sbom', 't09-gw:v1')
        tool = 'docker scout sbom'
    if rc != 0:
        print('skipped: SBOM-инструмент недоступен (docker sbom / docker scout) — '
              'риски заполните по Dockerfile и пинам, инвентарь не обязателен')
    else:
        with open(os.path.join(RESULTS, 'sbom-summary.txt'), 'w', encoding='utf-8') as fh:
            fh.write(out)
        _open = sum(1 for v in SUPPLY_RISKS.values() if str(v).startswith('TODO'))
        print(f'SBOM сохранён ({tool}): sbom-summary.txt ({len(out)} B); '
              f'незаполненных рисков: {_open}/3')

## Что вы построили — и чего эта лаба не доказывает

**Артефакт:** ваша производственная единица — pinned, non-root, healthchecked образ
`t09-gw:v1` с крошечной настоящей LLM (веса — mount), плюс журнал в
`results/local/attempt-001/`: замороженный план сборки с предсказаниями, три замера
скорости сборки, ведомость слоёв, журнал OpenAI-контракта через границу, запись
OOM-инцидента с кривой crash-loop, разбор диагностики и экономика рестарта. Журнал рос
вокруг одной вещи — единицы развёртывания, которая собирается быстро, весит мало и встаёт
после падения.

**Чему научились:** запускать и читать контейнеры в терминале; предсказывать инвалидацию
кэша до сборки; различать layer-кэш и cache-mount на реальной компиляции; вести
бухгалтерию размера по слоям и выбирать цену доставки весов (bake/mount/download);
проверять контракт через границу; разбирать прод-инцидент по коду выхода, журналу и
inspect; лечить crash-loop сайзингом и мерять рестарт-до-готовности именованными членами.

**Чего НЕ доказывает эта лаба:** секунды и мегабайты — ваши, локальные, машинно-зависимые
(курс не цитирует их как engine-бенчмарки); llama.cpp с 0.5B Q4 — это не vLLM с 7B:
паттерны продакшена честные, масштабы нет; настоящий Kubernetes не поднимался — только
его словарь на примитивах Docker; GPU-вариант образа — демонстрация владельца.

**Куда дальше:** Topic 10 масштабирует движки на несколько GPU; Topic 11 ставит эти образы
в Kubernetes, где pod, requests/limits, liveness/readiness и настоящий CrashLoopBackOff
станут ежедневностью. Ничего из этого не работает, если единица развёртывания не
неизменяема — чертёж вы теперь умеете писать, защищать и поднимать заново за секунды.

In [ ]:
artifacts = {}
for name in ('build-plan.json', 'build-timings.json', 'size-accounting.json',
             'contract-journal.json', 'oom-crashloop.json', 'restart-economics.json',
             'sbom-summary.txt'):
    path = os.path.join(RESULTS, name)
    if os.path.exists(path):
        with open(path, 'rb') as fh:
            artifacts[name] = hashlib.sha256(fh.read()).hexdigest()
with open(os.path.join(RESULTS, 'manifest.json'), 'w', encoding='utf-8') as fh:
    json.dump(dict(profile=PROFILE, attempt=ATTEMPT, artifacts=artifacts),
              fh, ensure_ascii=False, indent=1)
print(f'{len(artifacts)} артефактов хэшировано в manifest.json — журнал растущего артефакта закрыт')